In [13]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
import numpy as np

iris = load_iris()

iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['label'] = iris.target

print("Iris 데이터셋 미리보기:")
display(iris_df.head())
print("Iris 레이블 이름:", iris.target_names)

Iris 데이터셋 미리보기:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),label
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


Iris 레이블 이름: ['setosa' 'versicolor' 'virginica']


In [14]:
X = iris.data  # 피처 데이터
y = iris.target # 레이블 데이터

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=11, stratify=y)

print(f"학습 데이터 X_train 형태: {X_train.shape}")
print(f"테스트 데이터 X_test 형태: {X_test.shape}")
print(f"학습 레이블 y_train 형태: {y_train.shape}")
print(f"테스트 레이블 y_test 형태: {y_test.shape}")

print("\n학습 데이터 레이블 분포:\n", pd.Series(y_train).value_counts())
print("\n테스트 데이터 레이블 분포:\n", pd.Series(y_test).value_counts())

학습 데이터 X_train 형태: (120, 4)
테스트 데이터 X_test 형태: (30, 4)
학습 레이블 y_train 형태: (120,)
테스트 레이블 y_test 형태: (30,)

학습 데이터 레이블 분포:
 0    40
2    40
1    40
Name: count, dtype: int64

테스트 데이터 레이블 분포:
 1    10
0    10
2    10
Name: count, dtype: int64


In [15]:
dt_clf = DecisionTreeClassifier(random_state=11)

dt_clf.fit(X_train, y_train)
pred = dt_clf.predict(X_test)

accuracy = accuracy_score(y_test, pred)
print(f"결정 트리 모델의 예측 정확도: {accuracy:.4f}")

결정 트리 모델의 예측 정확도: 1.0000


In [16]:
print("\n--- KFold 교차 검증 (분류에서는 StratifiedKFold 권장) ---")
kfold = KFold(n_splits=5, shuffle=True, random_state=11) # 데이터를 섞고 시드 설정

fold_accuracies = []
for fold_num, (train_idx, test_idx) in enumerate(kfold.split(X), 1):
    X_train_fold, X_test_fold = X[train_idx], X[test_idx]
    y_train_fold, y_test_fold = y[train_idx], y[test_idx]

    dt_clf_fold = DecisionTreeClassifier(random_state=11)
    dt_clf_fold.fit(X_train_fold, y_train_fold)
    pred_fold = dt_clf_fold.predict(X_test_fold)
    accuracy_fold = accuracy_score(y_test_fold, pred_fold)
    fold_accuracies.append(accuracy_fold)
    print(f"KFold {fold_num} 정확도: {accuracy_fold:.4f}")
print(f"KFold 평균 정확도: {np.mean(fold_accuracies):.4f}")

print("\n--- StratifiedKFold 교차 검증 ---")
skfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=11)

sk_fold_accuracies = []
for fold_num, (train_idx, test_idx) in enumerate(skfold.split(X, y), 1): # y도 함께 전달하여 레이블 비율 유지
    X_train_skfold, X_test_skfold = X[train_idx], X[test_idx]
    y_train_skfold, y_test_skfold = y[train_idx], y[test_idx]

    dt_clf_skfold = DecisionTreeClassifier(random_state=11)
    dt_clf_skfold.fit(X_train_skfold, y_train_skfold)
    pred_skfold = dt_clf_skfold.predict(X_test_skfold)
    accuracy_skfold = accuracy_score(y_test_skfold, pred_skfold)
    sk_fold_accuracies.append(accuracy_skfold)
    print(f"StratifiedKFold {fold_num} 정확도: {accuracy_skfold:.4f}")
print(f"StratifiedKFold 평균 정확도: {np.mean(sk_fold_accuracies):.4f}")

print("\n--- cross_val_score 함수를 사용한 교차 검증 ---")
dt_clf_cv = DecisionTreeClassifier(random_state=11)
scores = cross_val_score(dt_clf_cv, X, y, scoring='accuracy', cv=3) # cv=3은 3-Fold

print('교차 검증별 정확도:', np.round(scores, 4))
print(f'평균 검증 정확도: {np.mean(scores):.4f}')


--- KFold 교차 검증 (분류에서는 StratifiedKFold 권장) ---
KFold 1 정확도: 0.9333
KFold 2 정확도: 0.9000
KFold 3 정확도: 1.0000
KFold 4 정확도: 0.9667
KFold 5 정확도: 0.9333
KFold 평균 정확도: 0.9467

--- StratifiedKFold 교차 검증 ---
StratifiedKFold 1 정확도: 0.9600
StratifiedKFold 2 정확도: 0.9000
StratifiedKFold 3 정확도: 0.9800
StratifiedKFold 평균 정확도: 0.9467

--- cross_val_score 함수를 사용한 교차 검증 ---
교차 검증별 정확도: [0.98 0.92 0.98]
평균 검증 정확도: 0.9600


In [10]:
params = {
    'max_depth': [1, 2, 3, 4, 5],
    'min_samples_split': [2, 3, 4]
}

dt_clf_grid = DecisionTreeClassifier(random_state=11)
grid_cv = GridSearchCV(dt_clf_grid, param_grid=params, cv=3, refit=True)
grid_cv.fit(X_train, y_train)

print('최적 하이퍼파라미터:', grid_cv.best_params_)
print(f'최고 교차 검증 정확도: {grid_cv.best_score_:.4f}')

best_dt_clf = grid_cv.best_estimator_ # 최적 모델 추출
pred_best = best_dt_clf.predict(X_test)
accuracy_best = accuracy_score(y_test, pred_best)
print(f"최적 모델의 테스트 세트 정확도: {accuracy_best:.4f}")

최적 하이퍼파라미터: {'max_depth': 3, 'min_samples_split': 2}
최고 교차 검증 정확도: 0.9583
최적 모델의 테스트 세트 정확도: 1.0000


In [17]:
print("\n--- 레이블 인코딩 (Label Encoding) ---")
items = ['TV', '냉장고', '전자레인지', '컴퓨터', '냉장고', 'TV']

encoder = LabelEncoder()

encoder.fit(items)
encoded_labels = encoder.transform(items)

print("원본 데이터:", items)
print("인코딩된 데이터:", encoded_labels)
print(f"인코딩 클래스 (알파벳 순):\n{encoder.classes_}")


--- 레이블 인코딩 (Label Encoding) ---
원본 데이터: ['TV', '냉장고', '전자레인지', '컴퓨터', '냉장고', 'TV']
인코딩된 데이터: [0 1 2 3 1 0]
인코딩 클래스 (알파벳 순):
['TV' '냉장고' '전자레인지' '컴퓨터']


In [12]:
print("\n--- 원-핫 인코딩 (One-Hot Encoding) ---")
items_2d = np.array(items).reshape(-1, 1)
ohe = OneHotEncoder(sparse_output=False) # sparse_output=False로 설정하여 밀집 행렬 반환
ohe.fit(items_2d)
encoded_ohe = ohe.transform(items_2d)

print(f"원본 데이터 (2D):\n{items_2d}")
print(f"원-핫 인코딩된 데이터 (밀집 행렬):\n{encoded_ohe}")
print("원-핫 인코딩 피처 이름:", ohe.get_feature_names_out())


--- 원-핫 인코딩 (One-Hot Encoding) ---
원본 데이터 (2D):
[['TV']
 ['냉장고']
 ['전자레인지']
 ['컴퓨터']
 ['냉장고']
 ['TV']]
원-핫 인코딩된 데이터 (밀집 행렬):
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]
원-핫 인코딩 피처 이름: ['x0_TV' 'x0_냉장고' 'x0_전자레인지' 'x0_컴퓨터']


In [8]:
print("\n--- 피처 스케일링 (Feature Scaling) ---")

print("원본 X_train (일부):\n", X_train[:5])
print("원본 X_test (일부):\n", X_test[:5])

print("\n--- 표준화 (StandardScaler) ---")
scaler_ss = StandardScaler()

X_train_scaled_ss = scaler_ss.fit_transform(X_train)
X_test_scaled_ss = scaler_ss.transform(X_test)

print("표준화된 X_train (일부):\n", np.round(X_train_scaled_ss[:5], 4))
print("표준화된 X_test (일부):\n", np.round(X_test_scaled_ss[:5], 4))

print("\n--- 정규화 (MinMaxScaler) ---")
scaler_mm = MinMaxScaler()

X_train_scaled_mm = scaler_mm.fit_transform(X_train)
X_test_scaled_mm = scaler_mm.transform(X_test)

print("정규화된 X_train (일부):\n", np.round(X_train_scaled_mm[:5], 4))
print("정규화된 X_test (일부):\n", np.round(X_test_scaled_mm[:5], 4))


--- 피처 스케일링 (Feature Scaling) ---
원본 X_train (일부):
 [[4.4 2.9 1.4 0.2]
 [5.1 3.3 1.7 0.5]
 [5.8 2.7 5.1 1.9]
 [5.  3.3 1.4 0.2]
 [6.4 2.7 5.3 1.9]]
원본 X_test (일부):
 [[6.5 2.8 4.6 1.5]
 [5.2 3.5 1.5 0.2]
 [5.8 2.7 4.1 1. ]
 [6.5 3.  5.8 2.2]
 [6.9 3.1 5.4 2.1]]

--- 표준화 (StandardScaler) ---
표준화된 X_train (일부):
 [[-1.7342 -0.3509 -1.3442 -1.3294]
 [-0.8892  0.5745 -1.1742 -0.9322]
 [-0.0443 -0.8136  0.7521  0.9212]
 [-1.0099  0.5745 -1.3442 -1.3294]
 [ 0.68   -0.8136  0.8654  0.9212]]
표준화된 X_test (일부):
 [[ 0.8007 -0.5822  0.4688  0.3916]
 [-0.7685  1.0372 -1.2875 -1.3294]
 [-0.0443 -0.8136  0.1856 -0.2703]
 [ 0.8007 -0.1195  1.1487  1.3184]
 [ 1.2835  0.1118  0.9221  1.186 ]]

--- 정규화 (MinMaxScaler) ---
정규화된 X_train (일부):
 [[0.     0.375  0.0678 0.0417]
 [0.2    0.5417 0.1186 0.1667]
 [0.4    0.2917 0.6949 0.75  ]
 [0.1714 0.5417 0.0678 0.0417]
 [0.5714 0.2917 0.7288 0.75  ]]
정규화된 X_test (일부):
 [[0.6    0.3333 0.6102 0.5833]
 [0.2286 0.625  0.0847 0.0417]
 [0.4    0.2917 0.5254 0.375 ]
 